In [12]:
import os, json
import cv2
import numpy as np
from tensorflow.keras.models import load_model

IMG_SIZE = 224
MODEL_PATH = "/app/models/best_model.h5"

data = np.load("/app/data/processed/dataset.npz")
class_names = [str(c) for c in data["class_named"]]

with open("/app/models/class_names.json", "w") as f:
    json.dump(class_names, f)

model = load_model(MODEL_PATH)
print(class_names)

['glioma', 'meningioma', 'notumor', 'pituitary']


In [13]:
def predict_image(path):
    image = cv2.imread(path)
    if image is None:
        raise ValueError(f"Could not read image: {path}")

    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    image = image.astype("float32") / 255.0
    image = np.expand_dims(image, axis=0)

    probs = model.predict(image, verbose=0)[0]
    idx = int(np.argmax(probs))

    return {
        "class": class_names[idx],
        "confidence": float(probs[idx]),
        "probabilities": {name: float(p) for name, p in zip(class_names, probs)}
    }

In [14]:
for cls in class_names:
    folder = os.path.join("/app/data", cls)
    path = os.path.join(folder, sorted(os.listdir(folder))[0])
    result = predict_image(path)
    print(f"True: {cls:<11} -> Predicted: {result['class']:<11} ({result['confidence']*100:.1f}%)")

try:
    predict_image("/app/data/does_not_exist.jpg")
except ValueError as e:
    print("Error handled:", e)

True: glioma      -> Predicted: glioma      (99.9%)
True: meningioma  -> Predicted: meningioma  (98.1%)
True: notumor     -> Predicted: notumor     (100.0%)
True: pituitary   -> Predicted: pituitary   (100.0%)
Error handled: Could not read image: /app/data/does_not_exist.jpg


[ WARN:0@15806.825] global loadsave.cpp:278 findDecoder imread_('/app/data/does_not_exist.jpg'): can't open/read file: check file path/integrity
